# Analyse des iteractions

Dans cette section nous nous intéresserons principalement aux données de type 'quote' ou 'comment' car ce sont ces lignes qui sont relatives à une intéraction entre deux comptes. En effet, les données de types original apporte des informations sur le tweet lui même.

# I. Chargement des données

In [ ]:
from Modules.data_preprocessing import DataPreprocessing
import os

dp = DataPreprocessing()
source_path = "."
df_path = os.path
df = dp.read_data("post_rehydrated.pickle", format_="pickle")
dp.parse_dates()
df.dtypes

In [ ]:
# Code lancé en local sur ma machine (lancer le code au dessus si sur la VM)
from Modules.data_preprocessing import DataPreprocessing

dp = DataPreprocessing()
df = dp.read_data("../post_rehydrated.pickle", format_="pickle")
dp.parse_dates()
df.dtypes

# II. Analyse des données

### 1. Répartition des observations selon le type d'intéraction

In [ ]:
df.groupby("join_post_post_type").agg(total=("join_post_post_type", "count"))

On peut noter que les commentaire sont les représenter dans notre base de données. Nous allons filtrer la table pour ne pas considérer les 'original'

In [ ]:
df_interac = df.loc[df["join_post_post_type"] != "original"]

In [ ]:
from Modules.interaction_analyzer_Ousseynou import InteractionAnalyzer

int_analyzer = InteractionAnalyzer(df=df_interac)

### 2. Etudions les les comptes qui intéragisse le plus dans la base de données

In [ ]:
repeated_amplifiers = int_analyzer.repeated_amplifiers(
    source_account_ids=[
        "1712592197160194048",
        "1720665183188922368",
        "1649752831497326593",
        "1290928302329409536",
        "38142665",
        "133663801",
    ],
    timeframe="2D",
).head(20)

repeated_amplifiers

### 2. Analyse du nombre de publication auxquelles réagit un utilisateur selon un fenêtre de temps

In [ ]:
int_analyzer.number_reaction().head(20)

On note que dans le jeu de données, les comptes qui réagissent le plus aux publications sont les comptes de `nicolasquipaie` et de `grok`. Pour plus observé les comptes qui ont une activité suspecte, nous allons réaliser une analyse incluant une fenêtre de temps

In [ ]:
number_reaction = int_analyzer.number_reaction(timeframe="1D").head(20)

number_reaction

Ainsi , lorsqu'on observe par exemple de nombre de citation et commentaire réalisé par jour, il ressort que `nicolasquipaie` et `grok` sont des comptes qui ont une activité très élévé par jour dans l'ensemble. De plus, en rélisant ces résultats au résultats précédent, on se propose de s'intéresser à certains compte en particulier.

In [ ]:
# Retrouvons les indentifiants tweeter suspectés plus haut dans les analyses

df.loc[
    (
        df["source_pf_account_id"].isin(
            [
                "1712592197160194048",
                "1720665183188922368",
                "1649752831497326593",
                "1290928302329409536",
                "38142665",
                "133663801",
            ]
        )
        & (df["join_post_post_type"] == "original")
    )
][["source_pf_account_id", "pf_account_id"]].drop_duplicates()

# On note que parmis les comptes sources suspecté, les comptes d'id 38142665 et 133663801 ne réagit jamais à une publication

In [ ]:
number_reaction = int_analyzer.number_reaction(
    pf_account_ids=[
        "nicolasquipaie",
        "ddrdzch",
        "boulisquen",
        "grok",
        # Comptes suspectés aussi
        "pravdamane",
    ],
    timeframe="1D",
).head(50)

number_reaction

On peut ainsi voir cette activité très importante des comptes ci-dessus

### 3. Détection d'amplificateurs

Dans cette section, nous allons inclure une fenêtre temporelle pour mieux détecter les amplications de certaines publications. Ci- dessous, nous retrouvons le taux de duplication de réaction pour une publication selon un nombre de jour après la publication

In [ ]:
int_analyzer.amplification_intensity().head(50)
# Ici, cette fonction ne me parait pas correcte dans le sens ou 33 action (33 lignes) ne sont pas relatif à 1 post donc le ratio calculer n'est pas correcte

In [ ]:
int_analyzer.amplification_intensity(timeframe="3D").head(50)

Sur cette sortie, on note que certaines publications connaissent un nombre de réations très important pour très peu de compte réagissant. Par exemple, le ratio le plus important est obtenue pour la publication du compte `1474837313146527746`.

Ici, nous allons filtrer la table pour observer les résultats sur les comptes suspect détecter grâce aux première analyse

In [ ]:
amplification_intensity = int_analyzer.amplification_intensity(
    source_account_ids=[
        "1712592197160194048",
        "1720665183188922368",
        "1649752831497326593",
        "1290928302329409536",
        "38142665",
        "133663801",
    ]
).head(50)

### 3. Intéraction avec une même publication dans un labs de temps

ici, on veut identifier les comptes qui semble avoir une intéraction coordonnées. Cette coordination peut être caractérisé par les réactions d'un même c